#  Prédiction du Prix Immobilier — Saône-et-Loire (71)

---

##  Description du projet

Ce notebook construit un **pipeline complet de Machine Learning** pour prédire le prix de vente de biens immobiliers dans les 20 plus grandes communes de Saône-et-Loire, à partir des données officielles DVF publiées par l'État français.

---

##  Qu'est-ce que le DVF ?

**DVF = Demandes de Valeurs Foncières**

Base de données publique publiée par la **DGFiP** (Direction Générale des Finances Publiques). Elle recense **toutes les transactions immobilières** réalisées en France et enregistrées par les notaires.

### Ce que contient le DVF :
- La **date de vente** et le **prix réel** de la transaction (pas le prix demandé)
- L'**adresse** et la **commune** du bien
- Le **type de bien** : Maison, Appartement, Dépendance, Local commercial
- La **surface habitable** et la **surface du terrain**
- Le **nombre de pièces principales**
- Les **coordonnées GPS** (latitude / longitude)

### Où le télécharger :
→ https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/

### Avantages du DVF :
| Critère | DVF | Plateformes commerciales |
|:--------|:---:|:------------------------:|
| Prix **réels** vendus |  |  prix demandés (+5 à +15%) |
| Gratuit |  |  souvent payant |
| Officiel (État) |  |  données privées |
| Exhaustif (toutes les ventes) |  |  partiel |

---

##  Limites connues et documentées

### Limite 1 — Ce que le DVF ne contient pas
Ces informations **absentes du DVF** empêchent d'atteindre un R² > 0.75 :

| Information manquante | Impact estimé sur le prix |
|:----------------------|:--------------------------|
| **DPE** (Diagnostic de Performance Énergétique) | ±10 à 20% (bien F/G vs A/B) |
| **Étage** (appartements) | RDC < étages intermédiaires < dernier étage |
| **État général** du bien | ±20 à 40% (rénové vs travaux) |
| **Exposition** (nord/sud) | Impact luminosité et valeur |
| **Charges de copropriété** | Charges élevées = décote à la négociation |
| **Garage / parking inclus** | +5 000€ à +20 000€ selon la ville |
| **Vue et environnement immédiat** | Non quantifiable sans données |

### Limite 2 — L'absence de données par quartier (limite géographique majeure)

C'est la **limitation principale** de ce modèle.

Dans une ville comme Chalon-sur-Saône ou Mâcon, le prix au m² peut varier de **800€/m² à 2 500€/m²** selon le quartier :
- Centre historique, quartiers résidentiels côté Saône → prix élevés
- Périphérie, zones moins prisées → prix significativement plus bas

Le DVF contient latitude/longitude mais ces coordonnées sont **volontairement arrondies** pour protéger la vie privée des vendeurs. Il faudrait :
1. Croiser avec les **zonages IRIS** (maille géographique fine de l'INSEE)
2. Utiliser **GeoPandas** pour les jointures spatiales
3. Calculer des prix médians par IRIS depuis le DVF lui-même

→ **Conséquence pratique** : deux biens dans la même ville mais dans des quartiers différents recevront une estimation similaire, alors que leur prix réel peut différer de **30 à 50%**. La fourchette d'estimation (±MAE) capture une partie de cette incertitude.

### Limite 3 — Performances réalistes

| Type de bien | R² | MAE estimée | Contexte |
|:-------------|:--:|:-----------:|:---------|
| Appartement | ~0.69 | ~33 000€ | Bon pour des données DVF seules |
| Local commercial | ~0.66 | ~119 000€ | Prix très hétérogènes |
| Maison | ~0.59 | ~37 000€ | Manque DPE et état du bien |
| Dépendance | ~0.25 | ~31 000€ | Prix dépend du bien principal |

**Plafond réaliste avec DVF seul : R² ≈ 0.70**
Les plateformes avec DPE + état + photos atteignent R² ≈ 0.80-0.85.

---

##  Feuille de route (18 cellules)

| # | Étape | Description |
|:-:|:------|:------------|
| 1 | Imports | Toutes les librairies nécessaires |
| 2 | Chargement DVF | Téléchargement 2021-2023 |
| 3 | Filtrage | Communes et types de biens |
| 4 | EDA | Exploration et visualisation |
| 5 | Nettoyage | Outliers et doublons |
| 6 | Séparation | Principal vs Dépendance |
| 7 | Feature Engineer | Transformer custom dans le Pipeline |
| 8 | Pipeline | Architecture complète |
| 9 | Split | Train/test sans data leakage |
| 10 | Évaluation | Fonction de mesure |
| 11 | Entraînement | Modèles de base |
| 12 | Tuning V1 | RandomizedSearchCV |
| 13 | Tuning V2 | Régularisation renforcée |
| 14 | Sélection | Meilleur V1 vs V2 |
| 15 | Résidus | Analyse des erreurs |
| 16 | SHAP | Explicabilité |
| 17 | Sauvegarde | Export joblib |
| 18 | Gradio | Interface interactive |


---
##  Cellule 1 — Imports


In [ ]:
# ============================================================
# PROJET  : Prédiction Prix Immobilier — Saône-et-Loire
# SOURCE  : DVF (Demandes Valeurs Foncières) data.gouv.fr
# DONNÉES : Années 2021, 2022, 2023 — Département 71
# ============================================================

# -- Manipulation des données --------------------------------
import pandas as pd
import numpy as np

# -- Visualisation -------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# -- Sauvegarde des modèles ----------------------------------
# joblib est recommandé par sklearn :
# plus rapide que pickle pour les tableaux numpy et les modèles ML
import joblib

# -- Sklearn : construction du pipeline ----------------------
from sklearn.pipeline import Pipeline
# Pipeline : enchaîne FE → Preprocessing → Modèle en un seul objet
from sklearn.compose import ColumnTransformer
# ColumnTransformer : applique un traitement différent selon le type de colonne
from sklearn.base import BaseEstimator, TransformerMixin
# BaseEstimator + TransformerMixin : classes mères pour créer
# un transformer custom compatible sklearn (méthodes fit/transform)

# -- Sklearn : preprocessing ---------------------------------
from sklearn.preprocessing import (
    RobustScaler,   # normalisation résistante aux outliers (médiane/IQR)
    OrdinalEncoder  # encodage des variables catégorielles avec ordre
)
from sklearn.impute import SimpleImputer
# SimpleImputer : remplace les NaN par la médiane ou le mode

# -- Sklearn : modèles de régression -------------------------
from sklearn.ensemble import RandomForestRegressor
# RandomForest : bagging — N arbres en parallèle, moyenne des prédictions
from xgboost import XGBRegressor
# XGBoost : boosting — arbres séquentiels, chacun corrige le précédent
# Généralement le meilleur sur données tabulaires

# -- Sklearn : évaluation et optimisation --------------------
from sklearn.model_selection import (
    train_test_split,    # séparation train/test
    RandomizedSearchCV   # optimisation aléatoire des hyperparamètres
    # (plus rapide que GridSearchCV qui teste toutes les combinaisons)
)
from sklearn.metrics import (
    mean_absolute_error,  # MAE  : erreur moyenne en €
    mean_squared_error,   # MSE  : erreur quadratique moyenne
    r2_score              # R²   : % de variance expliquée (1=parfait)
)

# -- Explicabilité des prédictions ---------------------------
import shap
# SHAP : explique pourquoi le modèle prédit tel prix pour tel bien

# -- Divers --------------------------------------------------
import warnings
warnings.filterwarnings('ignore')

# Graine aléatoire : garantit la reproductibilité des résultats
set_seed = 1204

print(' Librairies chargées avec succès')


---
##  Cellule 2 — Chargement des données DVF

On charge 3 années pour avoir un volume suffisant de transactions.
Département **71 = Saône-et-Loire**. Durée estimée : 1-3 minutes.


In [ ]:
# Les fichiers DVF sont hébergés sur data.gouv.fr
# Format : CSV compressé (.csv.gz), un fichier par année et département

print(' Téléchargement DVF 2021-2023 (département 71)...')

dfs = []
for annee in ['2021', '2022', '2023']:
    url = (
        f'https://files.data.gouv.fr/geo-dvf/'
        f'latest/csv/{annee}/departements/71.csv.gz'
    )
    print(f'  → {annee}...')
    df_temp = pd.read_csv(
        url,
        compression='gzip',
        low_memory=False  # évite les avertissements de types mixtes
    )
    df_temp['annee_mutation'] = annee
    dfs.append(df_temp)

df_dvf = pd.concat(dfs, ignore_index=True)

print(f'\n Données chargées : {df_dvf.shape[0]:,} transactions')

# IMPORTANT : toujours vérifier les valeurs EXACTES avant de filtrer
# Les noms de colonnes peuvent différer de ce qu'on attend
print('\n Types de biens (valeurs EXACTES dans le DVF) :')
print(df_dvf['type_local'].value_counts(dropna=False))

print('\n Types de mutations :')
print(df_dvf['nature_mutation'].value_counts())

print('\n Top 20 communes :')
print(df_dvf['nom_commune'].value_counts().head(20))


---
##  Cellule 3 — Filtrage et préparation initiale


In [ ]:
# -- 20 communes cibles (les plus grandes de Saône-et-Loire) -
communes_71 = [
    'Chalon-sur-Saône', 'Mâcon', 'Le Creusot',
    'Autun', 'Montceau-les-Mines', 'Paray-le-Monial',
    'Charnay-lès-Mâcon', 'Digoin', 'Louhans',
    'Tournus', 'Saint-Vallier', 'Bourbon-Lancy',
    'Chauffailles', 'Gueugnon', 'Châtenoy-le-Royal',
    'Saint-Germain-du-Bois', 'Saint-Marcel',
    'Chagny', 'Blanzy', 'Saint-Rémy'
]

# -- Types de biens avec leurs NOMS EXACTS dans le DVF -------
# ATTENTION : 'Local industriel. commercial ou assimilé' est le vrai nom
# (avec un point et pas de virgule) — erreur courante à éviter
types_biens = [
    'Maison',
    'Appartement',
    'Dépendance',
    'Local industriel. commercial ou assimilé'
]

# -- Filtrage principal --------------------------------------
df_sl = df_dvf[
    (df_dvf['nature_mutation'] == 'Vente') &
    (df_dvf['nom_commune'].isin(communes_71)) &
    (df_dvf['type_local'].isin(types_biens))
].copy()

# Renommage pour simplifier l'affichage
df_sl['type_local'] = df_sl['type_local'].replace({
    'Local industriel. commercial ou assimilé': 'Local commercial'
})

# Correction format prix : le DVF utilise des virgules décimales
# ex: '150000,5' → on convertit en float Python standard
df_sl['valeur_fonciere'] = (
    df_sl['valeur_fonciere']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# -- Sélection des colonnes utiles ---------------------------
df_sl = df_sl[[
    'valeur_fonciere',            # TARGET : prix de vente réel
    'type_local',                 # Maison / Appartement / ...
    'surface_reelle_bati',        # surface habitable (m²)
    'nombre_pieces_principales',  # nombre de pièces
    'surface_terrain',            # terrain (m²) — NaN si aucun
    'nom_commune',                # commune de vente
    'code_postal',                # code postal
    'date_mutation',              # date de la transaction
    'annee_mutation',             # année
    'longitude',                  # GPS —  arrondi dans le DVF
    'latitude'                    # GPS —  insuffisant pour le quartier
]].copy()

print(f' Shape après filtrage : {df_sl.shape}')
print('\n Répartition par type :')
print(df_sl['type_local'].value_counts())
print('\n Valeurs manquantes :')
print(df_sl.isnull().sum())


---
##  Cellule 4 — Exploration des données (EDA)


In [ ]:
# -- Statistiques descriptives par type ---------------------
print(' STATISTIQUES PAR TYPE DE BIEN')
print('━' * 55)

for type_bien in ['Maison', 'Appartement', 'Local commercial', 'Dépendance']:
    data = df_sl[df_sl['type_local'] == type_bien]['valeur_fonciere'].dropna()
    data = data[data > 0]
    print(f'\n {type_bien} ({len(data):,} transactions)')
    print(f'   Médiane : {data.median():>12,.0f} €')
    print(f'   Moyenne : {data.mean():>12,.0f} €  ← si >> médiane = outliers')
    print(f'   Min     : {data.min():>12,.0f} €  ← valeurs à 1€ = dons/erreurs')
    print(f'   Max     : {data.max():>12,.0f} €  ← valeurs aberrantes')

# -- Distribution des prix par type --------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (type_bien, couleur) in enumerate(zip(
    ['Maison', 'Appartement', 'Local commercial', 'Dépendance'],
    ['steelblue', 'salmon', 'green', 'orange']
)):
    data = df_sl[df_sl['type_local'] == type_bien]['valeur_fonciere'].dropna()
    data = data[(data > 0) & (data <= data.quantile(0.99))]
    axes[idx].hist(data, bins=50, color=couleur, edgecolor='white', alpha=0.8)
    axes[idx].axvline(data.median(), color='red', linestyle='--', linewidth=2,
                      label=f'Médiane : {data.median():,.0f}€')
    axes[idx].set_title(f'{type_bien} ({len(data):,} ventes)')
    axes[idx].set_xlabel('Prix (€)')
    axes[idx].legend()
    axes[idx].xaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}k€')
    )

plt.suptitle('Distribution des prix — Saône-et-Loire 2021-2023', fontsize=14)
plt.tight_layout()
plt.show()

# -- Heatmap prix médian par commune -------------------------
# Note : un chiffre par commune cache les disparités entre quartiers
pivot = df_sl.groupby(['nom_commune', 'type_local'])['valeur_fonciere'].median().unstack(fill_value=0)
pivot = (pivot / 1000).round(0)
pivot = pivot.sort_values(pivot.columns[0], ascending=False)

plt.figure(figsize=(12, 10))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn', linewidths=0.5,
            cbar_kws={'label': 'Prix médian (k€)'})
plt.title('Prix médian par commune et type (k€)\n Un seul chiffre par commune — ne reflète pas les disparités de quartiers',
          fontsize=12)
plt.tight_layout()
plt.show()


---
##  Cellule 5 — Nettoyage des outliers

On supprime les prix aberrants : ventes symboliques à 1€ (donations déguisées), erreurs de saisie notariale.
**Méthode** : bornes réalistes par type + IQR 2%-98%.


In [ ]:
# -- Prix min/max réalistes pour la Saône-et-Loire -----------
prix_limites = {
    'Maison'          : (5_000,   1_500_000),
    'Appartement'     : (3_000,     800_000),
    'Local commercial': (5_000,   3_000_000),
    'Dépendance'      : (500,       200_000)
}

dfs_propres = []
print(' Nettoyage outliers par type :')
print('━' * 55)

for type_bien, (p_min, p_max) in prix_limites.items():
    df_type  = df_sl[df_sl['type_local'] == type_bien].copy()
    nb_avant = len(df_type)

    # Étape 1 : bornes réalistes
    df_type = df_type[
        (df_type['valeur_fonciere'] >= p_min) &
        (df_type['valeur_fonciere'] <= p_max)
    ]
    # Étape 2 : IQR 2%-98% — garde 96% des données au centre
    q_bas  = df_type['valeur_fonciere'].quantile(0.02)
    q_haut = df_type['valeur_fonciere'].quantile(0.98)
    df_type = df_type[
        (df_type['valeur_fonciere'] >= q_bas) &
        (df_type['valeur_fonciere'] <= q_haut)
    ]

    print(f'\n{type_bien}')
    print(f'  Avant     : {nb_avant:>6,}')
    print(f'  Après     : {len(df_type):>6,}')
    print(f'  Supprimés : {nb_avant - len(df_type):>6,}')
    print(f'  Médiane   : {df_type["valeur_fonciere"].median():>10,.0f} €')
    dfs_propres.append(df_type)

df_sl = pd.concat(dfs_propres, ignore_index=True)

# Suppression des doublons (même transaction enregistrée deux fois)
nb_avant = len(df_sl)
df_sl    = df_sl.drop_duplicates().reset_index(drop=True)

print(f'\n Doublons supprimés : {nb_avant - len(df_sl)}')
print(f' Shape final        : {df_sl.shape}')


---
##  Cellule 6 — Séparation Dépendance / Principal

La Dépendance a **100% de surface manquante** (un garage n'a pas de surface habitable).
→ 2 datasets avec features différentes → 2 modèles distincts.


In [ ]:
# Dataset PRINCIPAL : Maison, Appartement, Local commercial
# Features riches : surface, pièces, prix/m², ratios...
df_principal = df_sl[
    df_sl['type_local'] != 'Dépendance'
].copy().reset_index(drop=True)

# Dataset DÉPENDANCE : garages, caves, parkings
# 100% de surface_reelle_bati manquante
# R² attendu faible (~0.25) : le prix dépend souvent du bien principal
# vendu avec la dépendance → hors contrôle du modèle
df_dependance = df_sl[
    df_sl['type_local'] == 'Dépendance'
].copy().reset_index(drop=True)

pct = df_dependance['surface_reelle_bati'].isnull().mean() * 100
print(f' Principal  : {df_principal.shape}')
print(f'   {df_principal["type_local"].value_counts().to_dict()}')
print(f'\n Dépendance : {df_dependance.shape}')
print(f'   surface_reelle_bati manquante : {pct:.0f}% → modèle séparé justifié')


---
##  Cellule 7 — Feature Engineering dans le Pipeline

### Pourquoi dans le Pipeline et pas avant ?
Si on calcule `prix_m2_commune` avant le split → data leakage (le test influence le train) 
Dans le Pipeline : `fit()` calcule sur le TRAIN uniquement 

### Pourquoi pas de feature par quartier ?
Les coordonnées GPS du DVF sont **arrondies** pour protéger la vie privée.
Pour les quartiers précis, il faudrait un travail de géocodage avancé
(GeoPandas + mailles IRIS de l'INSEE) — hors scope de ce projet.


In [ ]:
class ImmobilierFE(BaseEstimator, TransformerMixin):
    """
    Feature Engineering immobilier dans le Pipeline.

    fit()      : apprend les stats de marché sur le TRAIN uniquement
    transform(): applique sur train ET test proprement
    → Zéro data leakage garanti

     Limite géographique :
    Toutes les features de marché sont calculées à l'échelle
    de la COMMUNE, pas du QUARTIER. Dans les grandes villes,
    les disparités intra-communales ne sont pas capturées.
    """

    def fit(self, X, y=None):
        # Fréquence des transactions par commune
        self.freq_commune_ = X['nom_commune'].value_counts().to_dict()

        if y is not None:
            temp           = X.copy()
            temp['target'] = np.asarray(y).reshape(-1)

            # Prix médian par commune (depuis le train uniquement)
            self.med_commune_ = (
                temp.groupby('nom_commune')['target'].median().to_dict()
            )
            # Prix au m² médian par commune
            # → contexte marché, pas prix de la transaction courante
            temp['_m2'] = temp['target'] / temp['surface_reelle_bati'].fillna(1)
            self.m2_commune_ = (
                temp.groupby('nom_commune')['_m2'].median().to_dict()
            )
            self.m2_q25_ = (
                temp.groupby('nom_commune')['_m2'].quantile(0.25).to_dict()
            )
            self.m2_q75_ = (
                temp.groupby('nom_commune')['_m2'].quantile(0.75).to_dict()
            )
        else:
            self.med_commune_ = {}
            self.m2_commune_  = {}
            self.m2_q25_      = {}
            self.m2_q75_      = {}
        return self

    def transform(self, X):
        X = X.copy()

        # 1. Temporel
        X['date_mutation'] = pd.to_datetime(X['date_mutation'])
        X['mois_vente']      = X['date_mutation'].dt.month
        X['trimestre_vente'] = X['date_mutation'].dt.quarter
        X['annee_mutation']  = X['date_mutation'].dt.year.astype(int)

        # 2. Surface terrain → 0 si absente (appartements)
        X['surface_terrain'] = X['surface_terrain'].fillna(0)
        surf = X['surface_reelle_bati'].fillna(1)

        # 3. Ratios de surface
        X['ratio_terrain_bati'] = X['surface_terrain'] / (surf + 1)
        X['surface_par_piece']  = surf / (X['nombre_pieces_principales'].fillna(1) + 1)

        # 4. Contexte marché local (stats apprises sur le train)
        m2_global  = np.median(list(self.m2_commune_.values()))  if self.m2_commune_  else 1500
        med_global = np.median(list(self.med_commune_.values())) if self.med_commune_ else 100_000

        X['commune_freq']    = X['nom_commune'].map(self.freq_commune_).fillna(1)
        X['prix_m2_commune'] = X['nom_commune'].map(self.m2_commune_).fillna(m2_global)
        X['prix_m2_q25']     = X['nom_commune'].map(self.m2_q25_).fillna(m2_global * 0.7)
        X['prix_m2_q75']     = X['nom_commune'].map(self.m2_q75_).fillna(m2_global * 1.3)
        X['ecart_marche']    = X['prix_m2_q75'] - X['prix_m2_q25']
        X['prix_med_commune']= X['nom_commune'].map(self.med_commune_).fillna(med_global)

        # 5. Features d'interaction
        X['prix_estime_brut']      = surf * X['prix_m2_commune']
        X['valeur_terrain_estime'] = X['surface_terrain'] * (X['prix_m2_commune'] / 3)
        X['score_liquidite']       = np.log1p(X['commune_freq'])

        # 6. Catégorie de commune (selon prix médian du train)
        def cat_commune(c):
            p = self.med_commune_.get(c, 0)
            if p >= 130_000: return 'premium'
            if p >= 90_000:  return 'intermediaire'
            return 'accessible'

        X['cat_commune'] = X['nom_commune'].apply(cat_commune)

        # 7. Catégorie de surface
        def cat_surface(s):
            if pd.isna(s):  return 'inconnue'
            if s < 40:      return 'tres_petit'
            if s < 70:      return 'petit'
            if s < 100:     return 'moyen'
            if s < 150:     return 'grand'
            return 'tres_grand'

        if 'surface_reelle_bati' in X.columns:
            X['cat_surface'] = X['surface_reelle_bati'].apply(cat_surface)

        # 8. Suppression colonnes inutiles
        X = X.drop(columns=[
            c for c in ['date_mutation', 'code_postal', 'type_local']
            if c in X.columns
        ])
        return X


class TargetEncoderCommune(BaseEstimator, TransformerMixin):
    """
    Encode nom_commune par le prix médian de la commune.
    Lissage bayésien : communes peu représentées → tire vers la moyenne globale.
    Plus informatif que OneHot (20 colonnes binaires → 1 colonne numérique).
    """

    def __init__(self, smoothing=10):
        self.smoothing = smoothing

    def _to_series(self, X):
        if isinstance(X, pd.DataFrame):
            if X.shape[1] != 1:
                raise ValueError('TargetEncoderCommune expects a single column.')
            return X.iloc[:, 0]
        if isinstance(X, pd.Series):
            return X
        arr = np.asarray(X)
        if arr.ndim == 2:
            if arr.shape[1] != 1:
                raise ValueError('TargetEncoderCommune expects a single column.')
            arr = arr[:, 0]
        return pd.Series(arr)

    def fit(self, X, y):
        commune = self._to_series(X)
        target = pd.Series(np.asarray(y).reshape(-1))
        self.global_mean_ = target.mean()
        stats = pd.DataFrame({'commune': commune, 'target': target})
        stats = stats.groupby('commune')['target'].agg(['mean', 'count'])
        n = stats['count']
        stats['encoded'] = (
            (n * stats['mean'] + self.smoothing * self.global_mean_) / (n + self.smoothing)
        )
        self.encoding_map_ = stats['encoded'].to_dict()
        return self

    def transform(self, X):
        commune = self._to_series(X)
        return commune.map(self.encoding_map_).fillna(self.global_mean_).to_numpy().reshape(-1, 1)


print(' ImmobilierFE et TargetEncoderCommune prêts')


---
##  Cellule 8 — Architecture du Pipeline


In [ ]:
# -- Features dataset PRINCIPAL (Maison, Appart, Local) ------
num_main = [
    'surface_reelle_bati', 'nombre_pieces_principales',
    'surface_terrain', 'longitude', 'latitude',
    'ratio_terrain_bati', 'surface_par_piece',
    'mois_vente', 'trimestre_vente', 'annee_mutation',
    'commune_freq', 'prix_m2_commune',
    'prix_m2_q25', 'prix_m2_q75', 'ecart_marche',
    'prix_med_commune', 'prix_estime_brut',
    'valeur_terrain_estime', 'score_liquidite'
]
# nom_commune → traité par TargetEncoderCommune
ord_main = ['cat_commune', 'cat_surface']

# -- Features dataset DÉPENDANCE -----------------------------
num_dep = [
    'surface_terrain', 'longitude', 'latitude',
    'mois_vente', 'trimestre_vente', 'annee_mutation',
    'commune_freq', 'prix_m2_commune',
    'prix_med_commune', 'score_liquidite', 'valeur_terrain_estime'
]
ord_dep = ['cat_commune']


def creer_pipeline(num_f, ord_f, modele):
    """
    Crée un pipeline sklearn complet :
    ImmobilierFE → ColumnTransformer → Modèle

    Structure du ColumnTransformer :
    - pipe_num     : Imputer(median) + RobustScaler
    - pipe_commune : Imputer(mode) + TargetEncoderCommune
    - pipe_ord     : Imputer(mode) + OrdinalEncoder
    """
    pipe_num = Pipeline([
        ('imp',    SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ])
    pipe_commune = Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', TargetEncoderCommune(smoothing=10))
    ])
    pipe_ord = Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])

    preprocessor = ColumnTransformer([
        ('num',     pipe_num,     num_f),
        ('commune', pipe_commune, ['nom_commune']),
        ('ord',     pipe_ord,     ord_f)
    ], remainder='drop')

    return Pipeline([
        ('fe',   ImmobilierFE()),
        ('prep', preprocessor),
        ('mod',  modele)
    ])


print(' Pipeline défini')
print(f'   Principal  : {len(num_main)} num + commune TE + {len(ord_main)} ord')
print(f'   Dépendance : {len(num_dep)} num + commune TE + {len(ord_dep)} ord')


---
##  Cellule 9 — Split train/test

**Règle absolue** : split **avant** tout preprocessing.
Le Pipeline garantit que le test n'influence jamais l'entraînement.


In [ ]:
splits = {}  # Format : splits[type_bien] = (X_tr, X_te, y_tr, y_te)

print('  SPLIT TRAIN/TEST (80% / 20%)')
print('━' * 50)

for type_bien in ['Maison', 'Appartement', 'Local commercial']:
    df_type = df_principal[df_principal['type_local'] == type_bien].copy()
    X = df_type.drop(columns=['valeur_fonciere'])
    y = df_type['valeur_fonciere']
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=set_seed
    )
    splits[type_bien] = (X_tr, X_te, y_tr, y_te)
    print(f'\n {type_bien}')
    print(f'   Train : {X_tr.shape[0]:>5,} | Test : {X_te.shape[0]:>4,}')
    print(f'   Prix médian train : {y_tr.median():>10,.0f} €')

X_dep = df_dependance.drop(columns=['valeur_fonciere'])
y_dep = df_dependance['valeur_fonciere']
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
    X_dep, y_dep, test_size=0.2, random_state=set_seed
)
splits['Dépendance'] = (X_tr_d, X_te_d, y_tr_d, y_te_d)
print(f'\n Dépendance : Train {X_tr_d.shape[0]:,} | Test {X_te_d.shape[0]:,}')


---
##  Cellule 10 — Fonction d'évaluation


In [ ]:
def evaluer(nom, pipeline, X_tr, X_te, y_tr, y_te):
    """Entraîne un pipeline et calcule MAE, RMSE, R², overfitting."""
    pipeline.fit(X_tr, y_tr)
    y_pred_tr = pipeline.predict(X_tr)
    y_pred_te = pipeline.predict(X_te)

    mae     = mean_absolute_error(y_te, y_pred_te)
    rmse    = np.sqrt(mean_squared_error(y_te, y_pred_te))
    r2_te   = r2_score(y_te, y_pred_te)
    r2_tr   = r2_score(y_tr, y_pred_tr)
    overfit = r2_tr - r2_te

    print(f'\n{"━" * 45}')
    print(f'  {nom}')
    print(f'{"━" * 45}')
    print(f'  MAE      : {mae:>12,.0f} €')
    print(f'  RMSE     : {rmse:>12,.0f} €')
    print(f'  R² Test  : {r2_te:>12.4f}')
    print(f'  R² Train : {r2_tr:>12.4f}')
    if overfit > 0.15:
        print(f'   Overfitting fort : {overfit:.3f}')
    elif overfit > 0.10:
        print(f'    Léger overfit   : {overfit:.3f}')
    else:
        print(f'   OK               : {overfit:.3f}')

    return {'Modèle': nom, 'MAE': round(mae, 0), 'RMSE': round(rmse, 0),
            'R² Test': round(r2_te, 4), 'R² Train': round(r2_tr, 4)}

print(' Fonction evaluer() prête')


---
##  Cellule 11 — Entraînement des modèles de base


In [ ]:
modeles_base = [
    ('Random Forest', RandomForestRegressor(
        n_estimators=100, max_depth=10, random_state=set_seed, n_jobs=-1
    )),
    ('XGBoost', XGBRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=6,
        random_state=set_seed, verbosity=0
    ))
]

resultats_base = {}
pipelines_base = {}

for type_bien in ['Maison', 'Appartement', 'Local commercial', 'Dépendance']:
    print(f'\n{"="*50}\n   {type_bien}\n{"="*50}')
    X_tr, X_te, y_tr, y_te = splits[type_bien]
    num_f = num_main if type_bien != 'Dépendance' else num_dep
    ord_f = ord_main if type_bien != 'Dépendance' else ord_dep

    res_type  = []
    best_r2   = -999
    best_pipe = None

    for nom, modele in modeles_base:
        pipe = creer_pipeline(num_f, ord_f, modele)
        res  = evaluer(f'{nom} [{type_bien}]', pipe, X_tr, X_te, y_tr, y_te)
        res_type.append(res)
        if res['R² Test'] > best_r2:
            best_r2   = res['R² Test']
            best_pipe = pipe

    resultats_base[type_bien] = res_type
    pipelines_base[type_bien] = best_pipe

print('\n Entraînement de base terminé')


---
##  Cellule 12 — Tuning V1 (RandomizedSearchCV)

20 combinaisons aléatoires × CV=5 folds. Durée : 5-15 minutes.


In [ ]:
param_xgb_v1 = {
    'mod__n_estimators'    : [100, 200, 300, 500],
    'mod__max_depth'       : [3, 4, 5, 6],
    'mod__learning_rate'   : [0.01, 0.05, 0.1],
    'mod__subsample'       : [0.6, 0.7, 0.8],
    'mod__colsample_bytree': [0.6, 0.7, 0.8],
    'mod__reg_alpha'       : [0, 0.1, 1, 5, 10],
    'mod__reg_lambda'      : [1, 5, 10, 20],
    'mod__min_child_weight': [1, 3, 5, 10]
}
param_rf_v1 = {
    'mod__n_estimators'     : [100, 200, 300],
    'mod__max_depth'        : [4, 6, 8, 10],
    'mod__min_samples_split': [10, 20, 30, 50],
    'mod__min_samples_leaf' : [5, 10, 20],
    'mod__max_features'     : ['sqrt', 0.5, 0.7]
}

pipelines_v1 = {}
resultats_v1 = {}

print('  TUNING V1 (20 itérations × CV=5)')

for type_bien in ['Maison', 'Appartement', 'Local commercial', 'Dépendance']:
    print(f'\n{"─"*50}\n   {type_bien}')
    X_tr, X_te, y_tr, y_te = splits[type_bien]
    num_f = num_main if type_bien != 'Dépendance' else num_dep
    ord_f = ord_main if type_bien != 'Dépendance' else ord_dep

    res_type = []
    best_r2  = -999
    best_pipe = None
    best_nom  = None

    for nom, ParamGrid, Modele in [
        ('XGBoost', param_xgb_v1, XGBRegressor(random_state=set_seed, verbosity=0)),
        ('RF',      param_rf_v1,  RandomForestRegressor(random_state=set_seed, n_jobs=-1))
    ]:
        pipe   = creer_pipeline(num_f, ord_f, Modele)
        search = RandomizedSearchCV(
            pipe, ParamGrid, n_iter=20, cv=5,
            scoring='r2', n_jobs=-1, random_state=set_seed, verbose=0
        )
        search.fit(X_tr, y_tr)
        res = evaluer(f'{nom} V1 [{type_bien}]',
                      search.best_estimator_, X_tr, X_te, y_tr, y_te)
        res_type.append(res)
        if res['R² Test'] > best_r2:
            best_r2   = res['R² Test']
            best_pipe = search.best_estimator_
            best_nom  = nom

    pipelines_v1[type_bien] = {
        'pipeline': best_pipe, 'nom': best_nom, 'r2': best_r2,
        'X_test': X_te, 'y_test': y_te, 'X_train': X_tr, 'y_train': y_tr
    }
    resultats_v1[type_bien] = res_type
    print(f'   Meilleur V1 : {best_nom} | R²={best_r2:.4f}')


---
##  Cellule 13 — Tuning V2 (régularisation renforcée)

V2 teste une régularisation plus agressive. Durée : 5-15 minutes.


In [ ]:
param_xgb_v2 = {
    'mod__n_estimators'    : [100, 200, 300, 500],
    'mod__max_depth'       : [2, 3, 4, 5],
    'mod__learning_rate'   : [0.01, 0.03, 0.05],
    'mod__subsample'       : [0.5, 0.6, 0.7],
    'mod__colsample_bytree': [0.5, 0.6, 0.7],
    'mod__reg_alpha'       : [0.1, 1, 5, 10, 20],
    'mod__reg_lambda'      : [5, 10, 20, 50],
    'mod__min_child_weight': [3, 5, 10, 20],
    'mod__gamma'           : [0, 0.1, 0.5, 1]
}
param_rf_v2 = {
    'mod__n_estimators'     : [100, 200, 300],
    'mod__max_depth'        : [3, 4, 5, 6],
    'mod__min_samples_split': [20, 30, 50, 100],
    'mod__min_samples_leaf' : [10, 20, 30],
    'mod__max_features'     : ['sqrt', 0.3, 0.5]
}

pipelines_v2 = {}
resultats_v2 = {}

print('  TUNING V2 — Régularisation renforcée (25 itérations × CV=5)')

for type_bien in ['Maison', 'Appartement', 'Local commercial', 'Dépendance']:
    print(f'\n{"─"*50}\n   {type_bien}')
    X_tr, X_te, y_tr, y_te = splits[type_bien]
    num_f = num_main if type_bien != 'Dépendance' else num_dep
    ord_f = ord_main if type_bien != 'Dépendance' else ord_dep

    res_type  = []
    best_r2   = -999
    best_pipe = None
    best_nom  = None

    for nom, ParamGrid, Modele in [
        ('XGBoost', param_xgb_v2, XGBRegressor(random_state=set_seed, verbosity=0)),
        ('RF',      param_rf_v2,  RandomForestRegressor(random_state=set_seed, n_jobs=-1))
    ]:
        pipe   = creer_pipeline(num_f, ord_f, Modele)
        search = RandomizedSearchCV(
            pipe, ParamGrid, n_iter=25, cv=5,
            scoring='r2', n_jobs=-1, random_state=set_seed, verbose=0
        )
        search.fit(X_tr, y_tr)
        res = evaluer(f'{nom} V2 [{type_bien}]',
                      search.best_estimator_, X_tr, X_te, y_tr, y_te)
        res_type.append(res)
        if res['R² Test'] > best_r2:
            best_r2   = res['R² Test']
            best_pipe = search.best_estimator_
            best_nom  = nom

    pipelines_v2[type_bien] = {
        'pipeline': best_pipe, 'nom': best_nom, 'r2': best_r2,
        'X_test': X_te, 'y_test': y_te, 'X_train': X_tr, 'y_train': y_tr
    }
    resultats_v2[type_bien] = res_type
    print(f'   Meilleur V2 : {best_nom} | R²={best_r2:.4f}')


---
##  Cellule 14 — Sélection finale : meilleur de V1 vs V2

**Enseignement de la comparaison** : V2 réduit l'overfitting mais au prix d'un R² Test plus faible.
C'est le **dilemme biais/variance** : on choisit toujours selon le **R² Test** (performance réelle).


In [ ]:
pipelines_finaux = {}

print(' SÉLECTION FINALE — V1 vs V2')
print('━' * 65)
print(f'{"Type":<22} {"R²V1":>7} {"R²V2":>7} {"Choix":>8} {"R² final":>10}')
print('━' * 65)

for type_bien in ['Maison', 'Appartement', 'Local commercial', 'Dépendance']:
    r2_v1 = pipelines_v1[type_bien]['r2']
    r2_v2 = pipelines_v2[type_bien]['r2']

    if r2_v1 >= r2_v2:
        pipelines_finaux[type_bien] = pipelines_v1[type_bien]
        choix, r2_f = 'V1', r2_v1
    else:
        pipelines_finaux[type_bien] = pipelines_v2[type_bien]
        choix, r2_f = 'V2', r2_v2

    print(f'{type_bien:<22} {r2_v1:>7.4f} {r2_v2:>7.4f} {choix:>8} {r2_f:>10.4f}')

print('\n Pipelines finaux sélectionnés')

# Visualisation
types_plot = ['Maison', 'Appartement', 'Local commercial', 'Dépendance']
r2_finals  = [pipelines_finaux[t]['r2'] for t in types_plot]
mae_finals = [
    mean_absolute_error(
        pipelines_finaux[t]['y_test'],
        pipelines_finaux[t]['pipeline'].predict(pipelines_finaux[t]['X_test'])
    ) for t in types_plot
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
couleurs  = ['steelblue', 'salmon', 'green', 'orange']

bars = axes[0].bar(types_plot, r2_finals, color=couleurs, alpha=0.85)
axes[0].axhline(0.7, color='red', linestyle='--', label='Seuil 0.7')
axes[0].set_ylabel('R² Test')
axes[0].set_title('R² final par type')
axes[0].legend()
for bar, val in zip(bars, r2_finals):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 val + 0.01, f'{val:.3f}', ha='center', fontsize=10)

bars2 = axes[1].bar(types_plot, mae_finals, color=couleurs, alpha=0.85)
axes[1].set_ylabel('MAE (€)')
axes[1].set_title('Erreur moyenne (MAE) par type')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}k€'))
for bar, val in zip(bars2, mae_finals):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 val + 500, f'{val/1000:.1f}k€', ha='center', fontsize=10)

plt.suptitle('Performance finale — Modèles sélectionnés', fontsize=13)
plt.tight_layout()
plt.show()


---
##  Cellule 15 — Analyse des résidus

Un bon modèle a des résidus **centrés sur zéro** et **aléatoirement distribués**.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 14))

print(' ANALYSE DES RÉSIDUS')
print('━' * 60)

for idx, type_bien in enumerate(['Maison', 'Appartement', 'Local commercial']):
    info   = pipelines_finaux[type_bien]
    y_pred = info['pipeline'].predict(info['X_test'])
    residu = info['y_test'].values - y_pred
    mape   = np.mean(np.abs(residu / info['y_test'].values)) * 100

    axes[idx, 0].scatter(y_pred, residu, alpha=0.3, color='steelblue', s=15)
    axes[idx, 0].axhline(0, color='red', linestyle='--', linewidth=2)
    axes[idx, 0].axhline(residu.mean(), color='orange', linestyle='--',
                          label=f'Moy: {residu.mean():,.0f}€')
    axes[idx, 0].set_title(f'{type_bien} — Résidus vs Prédictions')
    axes[idx, 0].set_xlabel('Prix prédit (€)')
    axes[idx, 0].set_ylabel('Résidu = Réel - Prédit (€)')
    axes[idx, 0].legend()

    axes[idx, 1].hist(residu, bins=50, color='salmon', edgecolor='white', alpha=0.8)
    axes[idx, 1].axvline(0, color='red', linestyle='--', linewidth=2)
    axes[idx, 1].set_title(f'{type_bien} — Distribution des résidus')
    axes[idx, 1].set_xlabel('Résidu (€)')

    print(f'\n {type_bien}')
    print(f'  Résidu moyen : {residu.mean():>10,.0f} € ← proche 0 = pas de biais')
    print(f'  MAPE         : {mape:>10.1f} %   ← erreur en % du prix')
    print(f'  % dans ±20k€ : {(np.abs(residu)<=20_000).mean()*100:.1f}%')
    print(f'  % dans ±50k€ : {(np.abs(residu)<=50_000).mean()*100:.1f}%')

plt.tight_layout()
plt.show()

print('\n  Note : les résidus élevés s\'expliquent souvent par')
print('    des différences de QUARTIER non capturées par le DVF.')


---
##  Cellule 16 — SHAP (Explicabilité)


In [ ]:
shap_data = {}

for type_bien in ['Maison', 'Appartement', 'Local commercial']:
    print(f'  → SHAP : {type_bien}...')
    info          = pipelines_finaux[type_bien]
    pipe          = info['pipeline']
    X_te          = info['X_test']
    X_transformed = pipe[:-1].transform(X_te)
    modele        = pipe.named_steps['mod']

    explainer   = shap.TreeExplainer(modele)
    shap_values = explainer.shap_values(X_transformed)

    # Noms features : num + commune TE (1 col) + ord
    names = (
        (num_main if type_bien != 'Dépendance' else num_dep) +
        ['commune_target_encoded'] +
        (ord_main if type_bien != 'Dépendance' else ord_dep)
    )
    shap_data[type_bien] = {
        'shap_values': shap_values, 'X_tr': X_transformed,
        'names': names, 'explainer': explainer
    }

print(' SHAP calculé')

# Importance globale
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for idx, type_bien in enumerate(['Maison', 'Appartement', 'Local commercial']):
    d = shap_data[type_bien]
    plt.sca(axes[idx])
    shap.summary_plot(
        d['shap_values'], d['X_tr'], feature_names=d['names'],
        plot_type='bar', show=False, max_display=10
    )
    axes[idx].set_title(f'Top 10 — {type_bien}', fontsize=11)

plt.suptitle('Importance SHAP par type de bien', fontsize=13)
plt.tight_layout()
plt.show()

print('\n TOP 3 FEATURES PAR TYPE')
for type_bien in ['Maison', 'Appartement', 'Local commercial']:
    d   = shap_data[type_bien]
    imp = np.abs(d['shap_values']).mean(axis=0)
    top = np.argsort(imp)[::-1][:3]
    print(f'\n {type_bien}')
    for rank, i in enumerate(top, 1):
        if i < len(d['names']):
            print(f'  {rank}. {d["names"][i]:<35} {imp[i]:>8,.0f} €')


---
##  Cellule 17 — Sauvegarde des pipelines


In [ ]:
prix_m2_ref = {}
MAE_FINALES = {}

for type_bien in ['Maison', 'Appartement', 'Local commercial']:
    df_type = df_principal[df_principal['type_local'] == type_bien].copy()
    df_type['prix_m2'] = df_type['valeur_fonciere'] / df_type['surface_reelle_bati']
    prix_m2_ref[type_bien] = df_type.groupby('nom_commune')['prix_m2'].median().to_dict()

    info    = pipelines_finaux[type_bien]
    y_pred  = info['pipeline'].predict(info['X_test'])
    mae     = mean_absolute_error(info['y_test'], y_pred)
    MAE_FINALES[type_bien] = int(mae)

print(' SAUVEGARDE')
print('━' * 55)

for type_bien in ['Maison', 'Appartement', 'Local commercial', 'Dépendance']:
    info        = pipelines_finaux[type_bien]
    nom_fichier = f'pipeline_final_{type_bien.lower().replace(" ", "_")}.joblib'
    joblib.dump({
        'pipeline': info['pipeline'], 'type_bien': type_bien,
        'r2': info['r2'], 'mae': MAE_FINALES.get(type_bien, 35000),
        'nom_modele': info['nom']
    }, nom_fichier, compress=3)
    print(f'   {nom_fichier} | R²={info["r2"]:.4f}')

joblib.dump(prix_m2_ref, 'prix_m2_reference.joblib')
joblib.dump(MAE_FINALES, 'mae_par_type.joblib')
print('   prix_m2_reference.joblib')
print('   mae_par_type.joblib')

print('\n BILAN FINAL DU PROJET')
print('═' * 65)
print(f'{"Type":<22} {"R²":>6} {"MAE":>12} {"Transactions":>14}')
print('─' * 65)
nb_tr = df_principal['type_local'].value_counts().to_dict()
nb_tr['Dépendance'] = len(df_dependance)
for tb in ['Appartement', 'Local commercial', 'Maison', 'Dépendance']:
    r2  = pipelines_finaux[tb]['r2']
    mae = MAE_FINALES.get(tb, 31000)
    n   = nb_tr.get(tb, 0)
    print(f'{tb:<22} {r2:>6.3f} {mae:>10,} € {n:>14,}')
print('═' * 65)
print('\n  Limites documentées :')
print('   1. Quartiers non modélisés → ±30% possible dans les grandes villes')
print('   2. Pas de DPE, étage, état, exposition, charges')
print('   3. Dépendance R²=0.25 — à utiliser avec précaution')


---
##  Cellule 18 — Interface Gradio

**Interface volontairement simple** : estimation centrale, fourchette basse/haute, récapitulatif.

La fourchette (±MAE) capture une partie de l'incertitude liée aux quartiers non modélisés.
Sur Colab, `share=True` génère un lien public valable 72h.
Pour un hébergement permanent → **Hugging Face Spaces** (gratuit).


In [ ]:
# !pip install gradio -q

import gradio as gr

pipelines_gradio = {}
for type_bien in ['Maison', 'Appartement', 'Local commercial']:
    nom  = f'pipeline_final_{type_bien.lower().replace(" ", "_")}.joblib'
    data = joblib.load(nom)
    pipelines_gradio[type_bien] = data['pipeline']
    print(f' {type_bien} | R²={data["r2"]:.3f} | MAE={data["mae"]:,}€')

prix_m2_ref = joblib.load('prix_m2_reference.joblib')
MAE_CHARGEE = joblib.load('mae_par_type.joblib')


def predire(type_bien, commune, surface_bati,
            nb_pieces, surface_terrain, mois, annee):
    """
    Prédit le prix et retourne :
    - Estimation centrale
    - Fourchette basse/haute (±MAE)
    - Récapitulatif des caractéristiques

    Note : la fourchette capture une partie de l'incertitude
    liée aux quartiers non modélisés dans le DVF.
    """
    try:
        bien = pd.DataFrame([{
            'type_local'               : type_bien,
            'nom_commune'              : commune,
            'surface_reelle_bati'      : float(surface_bati),
            'nombre_pieces_principales': float(nb_pieces),
            'surface_terrain'          : float(surface_terrain),
            'longitude'                : 4.5,
            'latitude'                 : 46.5,
            'date_mutation'            : f'{int(annee)}-{int(mois):02d}-01',
            'annee_mutation'           : str(int(annee)),
            'code_postal'              : '71000'
        }])

        prix  = max(0, pipelines_gradio[type_bien].predict(bien)[0])
        mae   = MAE_CHARGEE.get(type_bien, 40_000)
        prix_m2 = prix / float(surface_bati) if float(surface_bati) > 0 else 0
        nb_tr   = len(df_principal[df_principal['type_local'] == type_bien])

        return f"""
##  Estimation du prix

| À la baisse | **Estimation centrale** | À la hausse |
|:-----------:|:----------------------:|:-----------:|
| {max(0, prix - mae):,.0f} € | **{prix:,.0f} €** | {prix + mae:,.0f} € |

**Prix au m²** : {prix_m2:,.0f} €/m²

---

##  Récapitulatif

| Caractéristique | Valeur |
|:----------------|:-------|
| Type de bien | {type_bien} |
| Commune | {commune} |
| Surface habitable | {float(surface_bati):.0f} m² |
| Nombre de pièces | {float(nb_pieces):.0f} |
| Surface terrain | {float(surface_terrain):.0f} m² |
| Période | {int(mois):02d}/{int(annee)} |

---

*{nb_tr:,} transactions DVF 2021-2023 | Fourchette ±{mae:,}€
reflète notamment l'incertitude liée au quartier.*
        """
    except Exception as e:
        return f' Erreur : {str(e)}'


with gr.Blocks(
    title=' Estimateur Prix Immobilier — Saône-et-Loire',
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("""
    #  Estimateur de Prix Immobilier
    ## Saône-et-Loire — Données DVF officielles 2021-2023
    > Basé sur des transactions **réellement vendues** (DGFiP / data.gouv.fr)
    ---
    """)

    with gr.Row():
        with gr.Column():
            gr.Markdown('###  Localisation')
            type_bien = gr.Dropdown(
                choices=['Maison', 'Appartement', 'Local commercial'],
                value='Appartement', label='Type de bien'
            )
            commune = gr.Dropdown(
                choices=communes_71, value='Autun', label='Commune'
            )
            annee = gr.Slider(2021, 2024, value=2024, step=1, label='Année')
            mois  = gr.Slider(1, 12, value=6, step=1, label='Mois')

        with gr.Column():
            gr.Markdown('###  Caractéristiques')
            surface_bati = gr.Number(
                value=70, minimum=9, label='Surface habitable (m²)'
            )
            nb_pieces = gr.Slider(1, 10, value=3, step=1, label='Nombre de pièces')
            surface_terrain = gr.Number(
                value=0, minimum=0, label='Terrain (m²) — 0 si aucun'
            )

    gr.Markdown('---')
    btn    = gr.Button(' Estimer le prix', variant='primary', size='lg')
    output = gr.Markdown('_Remplis le formulaire et clique sur Estimer_')

    btn.click(
        fn=predire,
        inputs=[type_bien, commune, surface_bati,
                nb_pieces, surface_terrain, mois, annee],
        outputs=output
    )

    gr.Markdown('---\n###  Exemples')
    gr.Examples(
        examples=[
            ['Appartement', 'Autun',            65, 3,   0, 6, 2024],
            ['Maison',      'Chalon-sur-Saône', 100, 4, 300, 5, 2024],
            ['Appartement', 'Mâcon',             50, 2,   0, 3, 2024],
            ['Maison',      'Autun',             85, 4, 500, 9, 2024],
            ['Local commercial', 'Le Creusot',  120, 0,   0, 1, 2024],
        ],
        inputs=[type_bien, commune, surface_bati,
                nb_pieces, surface_terrain, mois, annee]
    )

    gr.Markdown("""
    ---
    ###  Limites du modèle
    - **Quartiers non modélisés** : dans une même ville, le prix peut varier de ±30% selon le quartier. Cette incertitude est partiellement capturée par la fourchette affichée.
    - **Données absentes du DVF** : DPE, étage, état du bien, exposition, charges de copropriété.
    - **Dépendances** non disponibles ici (R² = 0.25, estimations peu fiables).
    """)

# share=True → lien public 72h sur Colab
# Pour un lien permanent → Hugging Face Spaces (gratuit)
demo.launch(share=True, debug=True)


---
##  Cellule 19 — Création de la variable de scoring

### Principe du scoring immobilier

Le scoring répond à une question différente de la régression :

| Approche | Question | Usage |
|:---------|:---------|:------|
| **Régression** | *Combien vaut ce bien ?* | Estimation de prix |
| **Scoring** | *Ce bien est-il une bonne affaire ?* | Analyser une annonce |

### Comment on construit la variable cible

On utilise le **modèle de régression déjà entraîné** pour calculer un prix estimé pour chaque transaction historique du DVF. Ensuite on compare ce prix estimé au prix réel :

```
ratio = prix_réel / prix_estimé

ratio < 0.85   →  0  BONNE AFFAIRE  (vendu en dessous du marché)
ratio 0.85-1.15 →  1  PRIX MARCHÉ   (vendu au prix du marché)
ratio > 1.15   →  2  PAYÉ TROP CHER (vendu au dessus du marché)
```

### Usage en production

En production, l'utilisateur saisit le **prix affiché** d'une annonce :
```
Bien affiché à 110 000€  /  Modèle estime 145 000€
→ ratio = 110k / 145k = 0.76  →   BONNE AFFAIRE

Bien affiché à 160 000€  /  Modèle estime 145 000€
→ ratio = 160k / 145k = 1.10  →   PRIX MARCHÉ
```


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, RocCurveDisplay
)

# -- Imports supplémentaires pour le scoring -----------------
print(' Imports scoring chargés')

# -- Création de la variable cible pour chaque type ----------
# On utilise les pipelines de régression déjà entraînés
# pour prédire le prix estimé de chaque transaction historique

datasets_scoring = {}  # stockage par type de bien

# Seuils du scoring
SEUIL_BAS  = 0.85  # en dessous → bonne affaire
SEUIL_HAUT = 1.15  # au dessus  → trop cher

print('\n CRÉATION DE LA VARIABLE DE SCORING')
print('━' * 55)

for type_bien in ['Maison', 'Appartement', 'Local commercial']:

    # Données de ce type
    df_type = df_principal[
        df_principal['type_local'] == type_bien
    ].copy()

    X_type = df_type.drop(columns=['valeur_fonciere'])
    y_type = df_type['valeur_fonciere']

    # Prédiction du prix estimé via le modèle de régression
    # On utilise cross_val_predict pour éviter le data leakage :
    # chaque transaction est prédite par un modèle qui ne l'a PAS vue
    from sklearn.model_selection import cross_val_predict

    pipe_regr = pipelines_finaux[type_bien]['pipeline']
    prix_estime = cross_val_predict(
        pipe_regr, X_type, y_type,
        cv=5,      # 5 folds
        n_jobs=-1
    )

    # Calcul du ratio : prix réel / prix estimé
    ratio = y_type.values / np.maximum(prix_estime, 1)  # évite division par 0

    # Création de la variable cible 3 classes
    # 0 = BONNE AFFAIRE / 1 = PRIX MARCHÉ / 2 = TROP CHER
    y_score = np.where(
        ratio < SEUIL_BAS,  0,
        np.where(ratio <= SEUIL_HAUT, 1, 2)
    )

    print(f'\n {type_bien}')
    print(f'  Transactions totales : {len(y_score):,}')
    print(f'  0 = Bonne affaire    : {(y_score==0).sum():,} ({(y_score==0).mean()*100:.1f}%)')
    print(f'  1 = Prix marché      : {(y_score==1).sum():,} ({(y_score==1).mean()*100:.1f}%)')
    print(f'  2 = Trop cher        : {(y_score==2).sum():,} ({(y_score==2).mean()*100:.1f}%)')

    datasets_scoring[type_bien] = {
        'X': X_type,
        'y': y_score,
        'ratio': ratio,
        'prix_estime': prix_estime
    }

# -- Visualisation des distributions -------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
labels  = ['Bonne\naffaire', 'Prix\nmarché', 'Trop\ncher']
couleurs_score = ['green', 'orange', 'red']

for idx, type_bien in enumerate(['Maison', 'Appartement', 'Local commercial']):
    y_s    = datasets_scoring[type_bien]['y']
    counts = [np.sum(y_s == i) for i in range(3)]
    bars   = axes[idx].bar(labels, counts, color=couleurs_score, alpha=0.8, edgecolor='white')
    axes[idx].set_title(f'{type_bien}\n{len(y_s):,} transactions')
    axes[idx].set_ylabel('Nombre de transactions')
    for bar, val in zip(bars, counts):
        pct = val / len(y_s) * 100
        axes[idx].text(bar.get_x() + bar.get_width()/2,
                       val + 10, f'{pct:.1f}%', ha='center', fontsize=10)

plt.suptitle('Distribution des classes de scoring', fontsize=13)
plt.tight_layout()
plt.show()


---
##  Cellule 20 — Entraînement des modèles de scoring

On entraîne **3 modèles de classification** par type de bien.
La cible est **3 classes** : Bonne affaire (0) / Prix marché (1) / Trop cher (2).

**Métrique principale : AUC-ROC** en mode OvR (One vs Rest)
- Proche de 1.0 → le modèle distingue bien les classes
- 0.5 → aléatoire, inutile


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# Versions locales pour éviter toute ancienne classe gardée en mémoire par le kernel
class ScoringImmobilierFE(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.freq_commune_ = X['nom_commune'].value_counts().to_dict()

        if y is not None:
            temp = X.copy()
            temp['target'] = np.asarray(y).reshape(-1)
            self.med_commune_ = temp.groupby('nom_commune')['target'].median().to_dict()
            temp['_m2'] = temp['target'] / temp['surface_reelle_bati'].fillna(1)
            self.m2_commune_ = temp.groupby('nom_commune')['_m2'].median().to_dict()
            self.m2_q25_ = temp.groupby('nom_commune')['_m2'].quantile(0.25).to_dict()
            self.m2_q75_ = temp.groupby('nom_commune')['_m2'].quantile(0.75).to_dict()
        else:
            self.med_commune_ = {}
            self.m2_commune_ = {}
            self.m2_q25_ = {}
            self.m2_q75_ = {}
        return self

    def transform(self, X):
        X = X.copy()
        X['date_mutation'] = pd.to_datetime(X['date_mutation'])
        X['mois_vente'] = X['date_mutation'].dt.month
        X['trimestre_vente'] = X['date_mutation'].dt.quarter
        X['annee_mutation'] = X['date_mutation'].dt.year.astype(int)

        X['surface_terrain'] = X['surface_terrain'].fillna(0)
        surf = X['surface_reelle_bati'].fillna(1)
        X['ratio_terrain_bati'] = X['surface_terrain'] / (surf + 1)
        X['surface_par_piece'] = surf / (X['nombre_pieces_principales'].fillna(1) + 1)

        m2_global = np.median(list(self.m2_commune_.values())) if self.m2_commune_ else 1500
        med_global = np.median(list(self.med_commune_.values())) if self.med_commune_ else 100_000

        X['commune_freq'] = X['nom_commune'].map(self.freq_commune_).fillna(1)
        X['prix_m2_commune'] = X['nom_commune'].map(self.m2_commune_).fillna(m2_global)
        X['prix_m2_q25'] = X['nom_commune'].map(self.m2_q25_).fillna(m2_global * 0.7)
        X['prix_m2_q75'] = X['nom_commune'].map(self.m2_q75_).fillna(m2_global * 1.3)
        X['ecart_marche'] = X['prix_m2_q75'] - X['prix_m2_q25']
        X['prix_med_commune'] = X['nom_commune'].map(self.med_commune_).fillna(med_global)
        X['prix_estime_brut'] = surf * X['prix_m2_commune']
        X['valeur_terrain_estime'] = X['surface_terrain'] * (X['prix_m2_commune'] / 3)
        X['score_liquidite'] = np.log1p(X['commune_freq'])

        def cat_commune(commune):
            prix = self.med_commune_.get(commune, 0)
            if prix >= 130_000:
                return 'premium'
            if prix >= 90_000:
                return 'intermediaire'
            return 'accessible'

        def cat_surface(surface):
            if pd.isna(surface):
                return 'inconnue'
            if surface < 40:
                return 'tres_petit'
            if surface < 70:
                return 'petit'
            if surface < 100:
                return 'moyen'
            if surface < 150:
                return 'grand'
            return 'tres_grand'

        X['cat_commune'] = X['nom_commune'].apply(cat_commune)
        if 'surface_reelle_bati' in X.columns:
            X['cat_surface'] = X['surface_reelle_bati'].apply(cat_surface)

        return X.drop(columns=[c for c in ['date_mutation', 'code_postal', 'type_local'] if c in X.columns])


class ScoringTargetEncoderCommune(BaseEstimator, TransformerMixin):
    def __init__(self, smoothing=10):
        self.smoothing = smoothing

    def _to_series(self, X):
        if isinstance(X, pd.DataFrame):
            if X.shape[1] != 1:
                raise ValueError('ScoringTargetEncoderCommune expects a single column.')
            return X.iloc[:, 0]
        if isinstance(X, pd.Series):
            return X
        arr = np.asarray(X)
        if arr.ndim == 2:
            if arr.shape[1] != 1:
                raise ValueError('ScoringTargetEncoderCommune expects a single column.')
            arr = arr[:, 0]
        return pd.Series(arr)

    def fit(self, X, y):
        commune = self._to_series(X)
        target = pd.Series(np.asarray(y).reshape(-1))
        self.global_mean_ = target.mean()
        stats = pd.DataFrame({'commune': commune, 'target': target})
        stats = stats.groupby('commune')['target'].agg(['mean', 'count'])
        n = stats['count']
        stats['encoded'] = ((n * stats['mean'] + self.smoothing * self.global_mean_) / (n + self.smoothing))
        self.encoding_map_ = stats['encoded'].to_dict()
        return self

    def transform(self, X):
        commune = self._to_series(X)
        return commune.map(self.encoding_map_).fillna(self.global_mean_).to_numpy().reshape(-1, 1)

splits_scoring   = {}  # splits par type
resultats_scoring = {}  # résultats par type
pipelines_scoring = {}  # meilleur pipeline par type


def creer_pipeline_clf(num_f, ord_f, modele):
    """
    Pipeline de classification — même architecture que la régression.
    Seul le modèle final change (classifier au lieu de regressor).
    """
    pipe_num = Pipeline([
        ('imp',    SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ])
    pipe_commune = Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', ScoringTargetEncoderCommune(smoothing=10))
    ])
    pipe_ord = Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])
    preprocessor = ColumnTransformer([
        ('num',     pipe_num,     num_f),
        ('commune', pipe_commune, ['nom_commune']),
        ('ord',     pipe_ord,     ord_f)
    ], remainder='drop')
    return Pipeline([
        ('fe',   ScoringImmobilierFE()),
        ('prep', preprocessor),
        ('clf',  modele)
    ])


modeles_clf = [
    ('Logistic Regression', LogisticRegression(
        max_iter=1000, random_state=set_seed, class_weight='balanced'
    )),
    ('Random Forest', RandomForestClassifier(
        n_estimators=100, max_depth=8,
        random_state=set_seed, n_jobs=-1, class_weight='balanced'
    )),
    ('XGBoost', XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        random_state=set_seed, verbosity=0,
        eval_metric='mlogloss'
    ))
]

print(' ENTRAÎNEMENT DES MODÈLES DE SCORING')
print('=' * 55)

for type_bien in ['Maison', 'Appartement', 'Local commercial']:
    print(f'\n{"─"*55}')
    print(f'   {type_bien}')
    print(f'{"─"*55}')

    data  = datasets_scoring[type_bien]
    X_s   = data['X']
    y_s   = data['y']

    # Split stratifié pour garder les proportions de classes
    X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
        X_s, y_s, test_size=0.2,
        random_state=set_seed, stratify=y_s
    )
    splits_scoring[type_bien] = (X_tr_s, X_te_s, y_tr_s, y_te_s)

    print(f'  Train : {len(y_tr_s):,} | Test : {len(y_te_s):,}')

    num_f = num_main if type_bien != 'Dépendance' else num_dep
    ord_f = ord_main if type_bien != 'Dépendance' else ord_dep

    res_type  = []
    best_auc  = -999
    best_pipe = None
    best_nom  = None

    for nom, modele in modeles_clf:
        pipe = creer_pipeline_clf(num_f, ord_f, modele)
        pipe.fit(X_tr_s, y_tr_s)

        y_pred  = pipe.predict(X_te_s)
        y_proba = pipe.predict_proba(X_te_s)

        # AUC en mode OvR (One vs Rest) pour les 3 classes
        auc = roc_auc_score(y_te_s, y_proba, multi_class='ovr', average='macro')
        f1  = f1_score(y_te_s, y_pred, average='macro')
        acc = (y_pred == y_te_s).mean()

        print(f'\n  {nom}')
        print(f'  AUC (macro OvR) : {auc:.4f}')
        print(f'  F1  (macro)     : {f1:.4f}')
        print(f'  Accuracy        : {acc:.4f}')
        print(classification_report(y_te_s, y_pred,
              target_names=['Bonne affaire', 'Prix marché', 'Trop cher']))

        res_type.append({'Modèle': nom, 'AUC': round(auc, 4),
                         'F1': round(f1, 4), 'Accuracy': round(acc, 4)})

        if auc > best_auc:
            best_auc  = auc
            best_pipe = pipe
            best_nom  = nom

    resultats_scoring[type_bien] = res_type
    pipelines_scoring[type_bien] = {
        'pipeline': best_pipe, 'nom': best_nom, 'auc': best_auc,
        'X_test': X_te_s, 'y_test': y_te_s
    }
    print(f'\n   Meilleur : {best_nom} | AUC={best_auc:.4f}')


---
##  Cellule 21 — Évaluation et comparaison des modèles de scoring


In [ ]:
print(' CLASSEMENT FINAL — MODÈLES DE SCORING')
print('=' * 55)

for type_bien in ['Maison', 'Appartement', 'Local commercial']:
    df_res = pd.DataFrame(resultats_scoring[type_bien]).sort_values('AUC', ascending=False)
    info   = pipelines_scoring[type_bien]
    print(f'\n {type_bien}')
    print(df_res.to_string(index=False))
    print(f'  → Meilleur : {info["nom"]} | AUC={info["auc"]:.4f}')

# -- Matrice de confusion ------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
labels_cls = ['Bonne\naffaire', 'Prix\nmarché', 'Trop\ncher']

for idx, type_bien in enumerate(['Maison', 'Appartement', 'Local commercial']):
    info   = pipelines_scoring[type_bien]
    y_pred = info['pipeline'].predict(info['X_test'])
    cm     = confusion_matrix(info['y_test'], y_pred)

    # Normalisation par ligne (% de bien classés)
    cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]

    sns.heatmap(
        cm_norm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=labels_cls, yticklabels=labels_cls,
        ax=axes[idx], linewidths=0.5
    )
    axes[idx].set_xlabel('Prédit')
    axes[idx].set_ylabel('Réel')
    axes[idx].set_title(f'{type_bien}\nMatrice de confusion (normalisée)')

plt.suptitle('Matrices de confusion — Modèles de scoring', fontsize=13)
plt.tight_layout()
plt.show()

print('''
Comment lire la matrice de confusion :
  Diagonale    : biens correctement classés
  Hors diag.   : erreurs de classification
  Valeur 0.80  : 80% des biens de cette classe sont bien prédits
''')


---
##  Cellule 22 — Calibration des probabilités

### Pourquoi calibrer ?

Le modèle prédit des **probabilités** pour chaque classe :
```
→ 'Cette annonce a 78% de chance d\'être une bonne affaire'
```

Sans calibration, ces probabilités peuvent être trompeuses :
- Le modèle dit 80% mais la réalité est 60% → **mal calibré** 
- Le modèle dit 80% et la réalité est 79% → **bien calibré** 

En immobilier, une probabilité fiable est cruciale : un acheteur qui voit '78% de chance de bonne affaire' doit pouvoir s'y fier pour prendre une décision.


In [ ]:
pipelines_calibres = {}  # pipelines calibrés

print(' CALIBRATION DES PROBABILITÉS')
print('━' * 55)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, type_bien in enumerate(['Maison', 'Appartement', 'Local commercial']):
    info   = pipelines_scoring[type_bien]
    X_tr_s, X_te_s, y_tr_s, y_te_s = splits_scoring[type_bien]

    # Calibration isotonique — plus flexible que sigmoid
    # cv=5 : utilise une cross-validation pour éviter le leakage
    pipe_calib = CalibratedClassifierCV(
        info['pipeline'],
        method='isotonic',  # plus flexible que 'sigmoid'
        cv=5
    )
    pipe_calib.fit(X_tr_s, y_tr_s)

    # Comparaison AUC avant/après calibration
    # La calibration ne doit pas dégrader le classement
    auc_avant = info['auc']
    y_proba_calib = pipe_calib.predict_proba(X_te_s)
    auc_apres = roc_auc_score(
        y_te_s, y_proba_calib, multi_class='ovr', average='macro'
    )

    print(f'\n {type_bien}')
    print(f'  AUC avant calibration : {auc_avant:.4f}')
    print(f'  AUC après calibration : {auc_apres:.4f}')
    diff = auc_apres - auc_avant
    print(f'  Différence            : {"+" if diff >= 0 else ""}{diff:.4f}')

    # Courbe de calibration (classe 0 : bonne affaire)
    # Montre si les probabilités prédites correspondent à la réalité
    CalibrationDisplay.from_predictions(
        (y_te_s == 0).astype(int),
        y_proba_calib[:, 0],
        n_bins=8, ax=axes[idx], name=type_bien
    )
    axes[idx].plot([0, 1], [0, 1], 'k--', label='Calibration parfaite')
    axes[idx].set_title(f'{type_bien}\nCalibration (classe : Bonne affaire)')
    axes[idx].legend(fontsize=9)

    pipelines_calibres[type_bien] = {
        'pipeline': pipe_calib,
        'auc'     : auc_apres
    }

plt.suptitle(
    'Calibration des probabilités (avant vs ligne parfaite)\n'
    'Courbe proche de la diagonale = probabilités fiables',
    fontsize=12
)
plt.tight_layout()
plt.show()


---
##  Cellule 23 — SHAP pour le scoring

SHAP explique pourquoi le modèle de scoring attribue un certain score.
Ici on analyse la classe **'Bonne affaire' (0)** :
quelles features font qu'un bien est identifié comme sous-évalué ?


In [ ]:
shap_scoring = {}

for type_bien in ['Maison', 'Appartement', 'Local commercial']:
    print(f'  → SHAP scoring : {type_bien}...')

    info      = pipelines_scoring[type_bien]
    pipe      = info['pipeline']
    X_te_s    = info['X_test']

    # Transformation via pipeline sans le classifieur
    X_transformed = pipe[:-1].transform(X_te_s)

    # Extraction du classifieur XGBoost/RF
    clf = pipe.named_steps['clf']

    # TreeExplainer pour les modèles basés sur des arbres
    explainer   = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_transformed)

    num_f = num_main if type_bien != 'Dépendance' else num_dep
    ord_f = ord_main if type_bien != 'Dépendance' else ord_dep
    names = num_f + ['commune_target_encoded'] + ord_f

    shap_scoring[type_bien] = {
        'shap_values': shap_values,
        'X_tr'       : X_transformed,
        'names'      : names
    }

print(' SHAP scoring calculé')

# -- Summary plot pour la classe 0 (Bonne affaire) ----------
# shap_values[0] = contributions pour la classe 'Bonne affaire'
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, type_bien in enumerate(['Maison', 'Appartement', 'Local commercial']):
    d = shap_scoring[type_bien]
    plt.sca(axes[idx])

    # Si shap_values est une liste (multi-classes) → prendre classe 0
    sv = d['shap_values'][0] if isinstance(d['shap_values'], list) else d['shap_values']

    shap.summary_plot(
        sv, d['X_tr'], feature_names=d['names'],
        plot_type='bar', show=False, max_display=10
    )
    axes[idx].set_title(f'Top features — Bonne affaire\n{type_bien}', fontsize=11)

plt.suptitle(
    'SHAP Scoring — Quelles features identifient une bonne affaire ?',
    fontsize=13
)
plt.tight_layout()
plt.show()

# -- Top 3 features par type ---------------------------------
print('\n TOP 3 FEATURES POUR IDENTIFIER UNE BONNE AFFAIRE')
print('━' * 55)
for type_bien in ['Maison', 'Appartement', 'Local commercial']:
    d  = shap_scoring[type_bien]
    sv = d['shap_values'][0] if isinstance(d['shap_values'], list) else d['shap_values']
    imp = np.abs(sv).mean(axis=0)
    top = np.argsort(imp)[::-1][:3]
    print(f'\n {type_bien}')
    for rank, i in enumerate(top, 1):
        if i < len(d['names']):
            print(f'  {rank}. {d["names"][i]}')


---
##  Cellule 24 — Sauvegarde des modèles de scoring


In [ ]:
print(' SAUVEGARDE DES MODÈLES DE SCORING')
print('━' * 55)

for type_bien in ['Maison', 'Appartement', 'Local commercial']:
    info        = pipelines_calibres[type_bien]
    nom_fichier = f'scoring_{type_bien.lower().replace(" ", "_")}.joblib'

    joblib.dump({
        'pipeline'   : info['pipeline'],
        'type_bien'  : type_bien,
        'auc'        : info['auc'],
        'classes'    : {0: 'Bonne affaire', 1: 'Prix marché', 2: 'Trop cher'},
        'seuil_bas'  : SEUIL_BAS,
        'seuil_haut' : SEUIL_HAUT
    }, nom_fichier, compress=3)

    print(f'   {nom_fichier} | AUC={info["auc"]:.4f}')

print('\n BILAN SCORING')
print('═' * 50)
print(f'{"Type":<22} {"Modèle":<20} {"AUC":>6}')
print('─' * 50)
for type_bien in ['Maison', 'Appartement', 'Local commercial']:
    info = pipelines_scoring[type_bien]
    print(f'{type_bien:<22} {info["nom"]:<20} {info["auc"]:>6.4f}')
print('═' * 50)
print('''
Interprétation de l\'AUC pour le scoring :
  AUC > 0.85 → excellent
  AUC 0.75-0.85 → bon
  AUC 0.65-0.75 → correct
  AUC < 0.65    → trop d\'incertitude pour scorer de façon fiable
''')
